# Modelo ARX MIMO

Ressalta-se que a estrutura trabalhada neste trabalho é de um modelo ARX MIMO, isto é, um modelo em que há várias entradas e várias saídas, bem como trata-se de um modelo autoregressivo com entradas exógenas.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

arquivo = pd.read_csv('dados_suspensao.csv', decimal=',')
arquivo.head()

t = np.array(arquivo['t'])[::40]
# Entradas
#vr = np.array(arquivo['Vr'])[::40]
zr = np.array(arquivo['Zr'])[::40]
# Saídas
zs = np.array(arquivo['X2'])[::40]
zus = np.array(arquivo['X1'])[::40]



# Criando a figura com quatro subplot
fig = make_subplots(rows=2, cols=2)

# Adicionando os gráficos em cada um dos subplot
fig.add_trace(go.Scatter(x=t, y=zus, mode='markers'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=zs, mode='markers'), row=1, col=2)
fig.add_trace(go.Scatter(x=t, y=zr, mode='markers'), row=2, col=1)


fig.update_xaxes(title_text='t', row=1, col=1)
fig.update_xaxes(title_text='t', row=1, col=2)
fig.update_xaxes(title_text='t', row=2, col=1)
fig.update_xaxes(title_text='t', row=2, col=2)

fig.update_yaxes(title_text='$z_{us}(t)$',range=[min(zus),max(zus)],row=1, col=1)
fig.update_yaxes(title_text='$z_{s}(t)$',range=[min(zs),max(zs)],row=1, col=2)
fig.update_yaxes(title_text='$z_{r}(t)$',range=[min(zr),max(zr)], row=2, col=1)


# Ajustando o espaço entre os subplot
fig.update_layout(
    showlegend=False,
    template='plotly_white',
    margin=dict(l=50, r=50, t=50, b=50),
    grid=dict(rows=2, columns=2, pattern='independent'),
)

# Mostrando o gráfico
fig.show()



In [2]:
#@title Organização dos dados

# Estrutura do modelo sugerida
# zs(k) = theta1*zs(k-1) + theta2*zs(k-2) + theta3*zr(k-2) + theta4*vr(k-2)

n = 2

# Entradas do modelo
zusk_2 = np.array([zus[0:-2]]).T
zusk_1 = np.array([zus[1:-1]]).T
zsk_2 = np.array([zs[0:-2]]).T
zsk_1 = np.array([zs[1:-1]]).T
zrk_2 = np.array([zr[0:-2]]).T



# Saida
zsk = np.array([zs[2:]]).T


# Matrizes de regressão
X = np.concatenate((zusk_1,zsk_1),axis=1)
Y = zsk



In [3]:
from sklearn.linear_model import LinearRegression

# Criar o objeto
reg = LinearRegression(fit_intercept=False).fit(X, Y)

In [4]:
#@title Testando modelo
Ymodel = reg.predict(X) # Predição dos dados experimentais

# Parametros
theta = reg.coef_
print(theta)

# Coeficiente linear
bias = reg.intercept_
print(bias)

# Calculo do R2
R2 = reg.score(X, Y)
print(R2)


[[0.41156731 0.59453419]]
0.0
0.8890437788057499


In [5]:
#@title plotando modelo
tsim = np.array([t[2:]])

# Graficos
trace1 = go.Scatter(x=tsim[0,:], y=Y[:,0], name='Y(t)', yaxis='y1')
trace2 = go.Scatter(x=tsim[0,:], y=Ymodel[:,0], name='Ym(t)', yaxis='y1')



fig = go.Figure(data=[trace1,trace2])
fig.update_layout(
    title="Dados Reais vs. Modelo Preditivo para Zs",
    xaxis_title="t",
    yaxis_title="Zs(t)",
    template='plotly_white'
)
fig.show()

# Decisão acerca da melhor estrutura de modelo ARX

Verifica-se, portanto, que a estrutura do gráfico acima apresenta comportamento similar, mas com alguns desvios consideráveis.
Para decidir o melhor modelo ARX, lança-se mão de uma estrutura mais complexa e com disposição de mais memórias, bem como toma-se como base dois aspectos fundamentais em termos de decisão: complixidade e corretude. A complexidade é relativa à quantidade de dados regressivos tomados como base a fim de representar o modelo, podendo variar a valores altíssimos a depender da intenção e poder computacional disposto. Quando se trata de corretude, o que precisa ser levado em consideração é a proximidade do modelo ARX em relação ao valor 1 da estatística R².
Para decidir qual modelo utilizar, leva-se em consideração, então, o melhor balanço entre corretude e complexidade para decidir qual estrutura escolher.

In [6]:
#@title Procurando melhor modelo

# Variáveis existentes
# zusk_1, zsk_1, zrk_2, zsk_2

model_structures = []

# Loop para gerar as estruturas desejadas
for i in range(1, 7):
    if i == 1:
        structure = [zusk_1, zsk_1]
    elif i == 2:
        structure = [zusk_1, zsk_1, zrk_2]
    elif i == 3:
        structure = [zusk_1, zsk_1, zrk_2, zsk_2]
    elif i == 4:
        structure = [zrk_2, zsk_2]
    elif i == 5:
        structure = [zusk_1, zrk_2, zsk_2]
    elif i == 6:
        structure = [zusk_1, zsk_1, zrk_2]

    model_structures.append(structure)


In [7]:
from sklearn.linear_model import LinearRegression


# Loop sobre as estruturas de modelo
for i, structure in enumerate(model_structures, start=1):
    X = np.column_stack(structure)  # Empilhando as colunas para formar a matriz X

    # Criando o modelo de regressão linear
    reg = LinearRegression(fit_intercept=False).fit(X, zsk)
    print(f"Estrutura {i}")
    # Parâmetros do modelo
    theta = reg.coef_
    print(f"Parâmetros = {theta}")

    # Coeficiente linear
    bias = reg.intercept_
    print(f"Coeficiente linear = {bias}")

    # Cálculo do R2
    R2 = reg.score(X, zsk)
    print(f"R_quadratico = {R2}")


Estrutura 1
Parâmetros = [[0.41156731 0.59453419]]
Coeficiente linear = 0.0
R_quadratico = 0.8890437788057499
Estrutura 2
Parâmetros = [[ 0.84102765  0.51747466 -0.3299385 ]]
Coeficiente linear = 0.0
R_quadratico = 0.8930617850274781
Estrutura 3
Parâmetros = [[ 0.39321532  1.40267427  0.09830764 -0.91575368]]
Coeficiente linear = 0.0
R_quadratico = 0.9987602106906642
Estrutura 4
Parâmetros = [[0.7885544  0.14659063]]
Coeficiente linear = 0.0
R_quadratico = 0.7334866094719941
Estrutura 5
Parâmetros = [[ 2.10701489 -0.8182153  -0.20676893]]
Coeficiente linear = 0.0
R_quadratico = 0.8690893603393011
Estrutura 6
Parâmetros = [[ 0.84102765  0.51747466 -0.3299385 ]]
Coeficiente linear = 0.0
R_quadratico = 0.8930617850274781


In [8]:
#@title Achando a estrutura com o maior R quadrático

best_R2 = -1  # Valor inicial para encontrar o maior R²
best_model_index = -1  # Índice do melhor modelo
best_theta = None  # Parâmetros do melhor modelo
best_Ymodel = None  # Valores preditos do melhor modelo

# Loop sobre as estruturas de modelo
for i, structure in enumerate(model_structures, start=1):
    X = np.column_stack(structure)  # Empilhando as colunas para formar a matriz X

    # Criando o modelo de regressão linear
    reg = LinearRegression(fit_intercept=False).fit(X, zsk)  # Ajuste do modelo

    # Cálculo do R²
    R2 = reg.score(X, zsk)

    # Verificando se é o melhor R² encontrado até agora
    if R2 > best_R2:
        best_R2 = R2
        best_model_index = i
        best_theta = reg.coef_
        best_Ymodel = reg.predict(X)

# Selecione o modelo com o maior R² e plote
if best_model_index != -1:
    print(f"Melhor modelo: Estrutura {best_model_index}, R_quadratico = {best_R2}")



Melhor modelo: Estrutura 3, R_quadratico = 0.9987602106906642


In [9]:
#@title plotando o gráfico com melhor R quadrado
tsim = np.array([t[2:]])

# Grafico do melhor modelo com o maior R²
trace1 = go.Scatter(x=tsim[0, :], y=Y[:, 0], name='dados reais', yaxis='y1')
trace2 = go.Scatter(x=tsim[0, :], y=best_Ymodel[:, 0], name = 'Melhor modelo', yaxis='y1')

fig = go.Figure(data=[trace1, trace2])
fig.update_layout(
    title="Dados Reais vs. Modelo Preditivo otmizado para Zs",
    xaxis_title="t",
    yaxis_title="Zs(t)",
    template='plotly_white'
)
fig.show()

In [10]:
#@title modelo para ZUS
# Estrutura do modelo
# zus(k) = theta1*zus(k-1) + theta2*zus(k-2) + theta3*zr(k-2) + theta4*vr(k-2)

n = 2

# Entradas do modelo
zusk_2 = np.array([zus[0:-2]]).T
zusk_1 = np.array([zus[1:-1]]).T
zsk_2 = np.array([zs[0:-2]]).T
zsk_1 = np.array([zs[1:-1]]).T
zrk_2 = np.array([zr[0:-2]]).T


# Saída
zusk = np.array([zus[2:]]).T

# Matrizes de regressão
X = np.concatenate((zusk_1, zsk_1, zrk_2), axis=1)
Y = zusk

In [11]:
from sklearn.linear_model import LinearRegression
# Criar o objeto
reg = LinearRegression(fit_intercept=False).fit(X, Y)  # Testando modelo
Ymodel = reg.predict(X)  # Predição dos dados experimentais

In [12]:
# Parâmetros
theta = reg.coef_
print("Coeficientes:", theta)

# Cálculo do R2
R2 = reg.score(X, Y)
print("R2:", R2)

Coeficientes: [[ 1.24618099 -0.13776676 -0.12310889]]
R2: 0.9337690907794504


In [13]:
tsim = np.array([t[0:]])

trace1 = go.Scatter(x=tsim[0, :], y=Y[:, 0], name='Real', yaxis='y1')  # Dados reais
trace2 = go.Scatter(x=tsim[0, :], y=Ymodel[:, 0], name='Modelo', yaxis='y1')  # Modelo previsto

fig = go.Figure(data=[trace1, trace2])
fig.update_layout(
    title="Dados Reais vs. Modelo Preditivo para zus",
    xaxis_title="t",
    yaxis_title="zus(t)",
    template='plotly_white'
)
fig.show()

In [14]:
#@title Estrutura para achar modelos de forma automatizada

model_structures2 = []

# Loop para gerar as estruturas desejadas
for i in range(1, 7):
    if i == 1:
        structure = [zusk_1, zsk_1]
    elif i == 2:
        structure = [zusk_1, zsk_1, zrk_2]
    elif i == 3:
        structure = [zusk_1, zsk_1, zrk_2, zsk_2]
    elif i == 4:
        structure = [zrk_2, zsk_2]
    elif i == 5:
        structure = [zusk_1, zrk_2, zsk_2]
    elif i == 6:
        structure = [zusk_1, zsk_1, zrk_2]

    model_structures2.append(structure)


In [15]:
from sklearn.linear_model import LinearRegression


# Loop sobre as estruturas de modelo
for i, estrutura in enumerate(model_structures2, start=1):
    X = np.column_stack(estrutura)  # Empilhando as colunas para formar a matriz X

    # Criando o modelo de regressão linear
    reg = LinearRegression(fit_intercept=False).fit(X, zusk)
    print(f'Estrutura {i}')
    # Parâmetros do modelo
    theta = reg.coef_
    print(f'Parâmetros = {theta}')

    # Coeficiente linear
    bias = reg.intercept_
    print(f"Coeficiente linear = {bias}")

    # Cálculo do R2
    R2 = reg.score(X, zusk)
    print(f"R_quadrado = {R2}")

Estrutura 1
Parâmetros = [[ 1.08593784 -0.10901379]]
Coeficiente linear = 0.0
R_quadrado = 0.9329903224417062
Estrutura 2
Parâmetros = [[ 1.24618099 -0.13776676 -0.12310889]]
Coeficiente linear = 0.0
R_quadrado = 0.9337690907794504
Estrutura 3
Parâmetros = [[ 1.10192991  0.14737723  0.01483945 -0.2949862 ]]
Coeficiente linear = 0.0
R_quadrado = 0.9490376979762348
Estrutura 4
Parâmetros = [[ 0.89616796 -0.0054952 ]]
Coeficiente linear = 0.0
R_quadrado = 0.8771585630754711
Estrutura 5
Parâmetros = [[ 1.28199669 -0.08145847 -0.22049405]]
Coeficiente linear = 0.0
R_quadrado = 0.9470448491189916
Estrutura 6
Parâmetros = [[ 1.24618099 -0.13776676 -0.12310889]]
Coeficiente linear = 0.0
R_quadrado = 0.9337690907794504


In [16]:
best_R22 = -1  # Valor inicial para encontrar o maior R²
best_model_index2 = -1  # Índice do melhor modelo
best_theta2 = None  # Parâmetros do melhor modelo
best_Ymodel2 = None  # Valores preditos do melhor modelo

# Loop sobre as estruturas de modelo
for i, estrutura in enumerate(model_structures2, start=1):
    X = np.column_stack(estrutura)  # Empilhando as colunas para formar a matriz X

    # Criando o modelo de regressão linear
    reg = LinearRegression(fit_intercept=False).fit(X, zusk)  # Ajuste do modelo

    # Cálculo do R²
    R22 = reg.score(X, zusk)

    # Verificando se é o melhor R² encontrado até agora
    if R22 > best_R22:
        best_R22 = R22
        best_model_index2 = i
        best_theta2 = reg.coef_
        best_Ymodel2 = reg.predict(X)

# Selecione o modelo com o maior R quadrado
if best_model_index2 != -1:
    print(f"Melhor modelo: Estrutura {best_model_index2}, R_quadratico = {best_R22}")

Melhor modelo: Estrutura 3, R_quadratico = 0.9490376979762348


In [17]:
tsim2 = np.array([t[0:]])

# Grafico do melhor modelo com o maior R²
trace1 = go.Scatter(x=tsim2[0, :], y=Y[:, 0], name='Real', yaxis='y1')
trace2 = go.Scatter(x=tsim2[0, :], y=best_Ymodel2[:, 0], name='Melhor modelo', yaxis='y1')

fig = go.Figure(data=[trace1, trace2])
fig.update_layout(
    title="Dados Reais vs. Modelo Preditivo otimizado para zus",
    xaxis_title="t",
    yaxis_title="zus(t)",
    template='plotly_white'
)
fig.show()

# Comparação MODELO ARX vs modelo Fenomenológico otimizado da AV1

Considerando os modelos testados, vê-se que o mais complexo (representado pela estrutura 3) apresentou comportamento bem representativo frente aos dados reais. Dessa forma, toma-se como base para a representação do sistema, embora seja possível utilizar modelos ainda mais complexo, mas que eventualmente trariam mais complexidade ao algoritmo sem ganhos consideraveis no coeficiente R².

O modelo encontrado na AVI é provido de muitas informações acerca das leis físicas que regem o sistema. No entanto, a fim de encontrar uma representação com maior precisão em relação às saídas, entende-se como importante um processo de otimização dos parâmetros utilizados no sistema. Tal processo, enquanto benéfico à exatidão e à precisão, provoca um aumento grande em termos de processamento.
Tratando-se de um modelo baseado em ARX, isto é, que utiliza uma estrutura de comparação regressiva de dados para aquiusição de informações, é possível, a depender da ordem adotada, conseguir um bom grau de precisão em relação aos dados reais e com o possível fator positivo de menor complexidade computacional. No entanto, é válido ressaltar que aumentar a ordem do modelo ARX MIMO indefinidamente não necessariamente trará resultados eficientemente positivos, haja vista que o coeficiente estatístico R² (quanto mais perto de 1, melhor) para determinação de corretude do modelo pode não sofrer melhoras
consideráveis embora a complexidade do modelo, ao aumentar os dados regressivos que serão usados, pode aumentar consideravelmente.
